# Ejercicio 1. ANSI C

**Código**

A continuación se presenta la implementación del programa en lenguaje C, cuyo objetivo es generar un árbol de procesos con una estructura jerárquica específica.

In [ ]:
%%writefile process_tree.c

#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <sys/wait.h>
#include <string.h>
#include <linux/prctl.h>
#include <sys/prctl.h>

#define PROCESS_NAME_MAX_LENGTH 16
#define SLEEP_DURATION_SECONDS 30

// Number of children per parent process
#define NUM_CHILDREN_A 1  // B
#define NUM_CHILDREN_B 2  // C and D
#define NUM_CHILDREN_C 1  // E
#define NUM_CHILDREN_D 2  // F and G
#define NUM_CHILDREN_E 2  // H and I
#define NUM_CHILDREN_LEAF 0  // Leaf nodes: F, G, H, I

void handleChildError(const char *childName);
void setProcessName(const char *name);
void sleepAndWait(int numChildren);
void createLeaf(const char *leafName);

int main()
{
  pid_t pid = fork();

  if (pid < 0)
  {
    handleChildError("B");
    return EXIT_FAILURE;
  }

  if (pid > 0)
  {
    setProcessName("A");
    sleepAndWait(NUM_CHILDREN_A);
    return EXIT_SUCCESS;
  }

  // Process B
  setProcessName("B");

  pid = fork();
  if (pid < 0)
  {
    handleChildError("C");
    return EXIT_FAILURE;
  }

  if (pid == 0)
  {
    // Process C
    setProcessName("C");

    pid = fork();
    if (pid < 0)
    {
      handleChildError("E");
      return EXIT_FAILURE;
    }

    if (pid == 0)
    {
      // Process E
      setProcessName("E");

      createLeaf("H");
      createLeaf("I");

      sleepAndWait(NUM_CHILDREN_E);
      return EXIT_SUCCESS;
    }

    sleepAndWait(NUM_CHILDREN_C);
    return EXIT_SUCCESS;
  }

  pid = fork();
  if (pid < 0)
  {
    handleChildError("D");
    return EXIT_FAILURE;
  }

  if (pid == 0)
  {
    // Process D
    setProcessName("D");

    createLeaf("F");
    createLeaf("G");

    sleepAndWait(NUM_CHILDREN_D);
    return EXIT_SUCCESS;
  }

  sleepAndWait(NUM_CHILDREN_B);
  return EXIT_SUCCESS;
}

void handleChildError(const char *childName)
{
  fprintf(stderr, "Error creating process %s\n", childName);
}

void setProcessName(const char *name)
{
  char buffer[PROCESS_NAME_MAX_LENGTH];
  strncpy(buffer, name, PROCESS_NAME_MAX_LENGTH);
  prctl(PR_SET_NAME, buffer, 0, 0, 0);
}

void sleepAndWait(int numChildren)
{
  sleep(SLEEP_DURATION_SECONDS);
  for (int i = 0; i < numChildren; ++i)
  {
    wait(NULL);
  }
}

void createLeaf(const char *leafName)
{
  pid_t pid = fork();

  if (pid < 0)
  {
    handleChildError(leafName);
    exit(EXIT_FAILURE);
  }

  if (pid == 0)
  {
    setProcessName(leafName);
    sleepAndWait(NUM_CHILDREN_LEAF);
    exit(EXIT_SUCCESS);
  }
}

Writing process_tree.c




---


**Compilación**

Se compila el archivo fuente `process_tree.c` utilizando `gcc`, generando el ejecutable `process_tree`:

In [ ]:
!gcc process_tree.c -o process_tree



---


**Ejecución**

El programa se ejecuta en segundo plano usando `nohup`, redirigiendo tanto la salida estándar como la de error para evitar interferencias durante la visualización:

In [ ]:
!nohup ./process_tree 1>/dev/null 2>/dev/null &



---


**Visualización**

Finalmente, se utiliza el comando `pstree` junto con `pgrep` para capturar el PID del proceso raíz y visualizar el árbol completo:

In [ ]:
!pstree -pc $(pgrep -f "process_tree" | head -1)

A(1332)───B(1333)─┬─C(1334)───E(1336)─┬─H(1338)
                  │                   └─I(1340)
                  └─D(1335)─┬─F(1337)
                            └─G(1339)




---


# Ejercicio 2. Python

**a. Explicación del código original**

Lo que hace el siguiente programa es simular un juego de dados para varios jugadores usando procesos paralelos.

In [ ]:
from multiprocessing import Process
import random
import time
import sys

PLAYER = 5
THROWS = 10

def player(id):
    sys.stdout.write(f"Jugador {id} entra al juego.\n")
    points = 0
    for i in range(THROWS):
        dice = random.randint(1, 6)
        points += dice
        sys.stdout.write(f"Jugador {id} - Lanzamiento {i+1}: {dice}\n")
        time.sleep(random.uniform(0.1, 0.3))
    sys.stdout.write(f"Jugador {id} finaliza con {points} puntos.\n")

def main():
    procesos = []
    for i in range(PLAYER):
        p = Process(target=player, args=(i+1,))
        procesos.append(p)
        p.start()

    for p in procesos:
        p.join()

    print("Todos los jugadores han terminado.")

if __name__ == "__main__":
    main()

**b. Desarrollo utilizando fork() en lugar de Process.**

In [ ]:
%%writefile fork-processes.py

import random
import time
import sys
import os

PLAYER = 5
THROWS = 10
DICE_MIN = 1
DICE_MAX = 6
SLEEP_MIN = 0.1
SLEEP_MAX = 0.3
WAIT_BLOCKING = 0
THROW_INDEX_OFFSET = 1
PLAYER_ID_OFFSET = 1

def player(id):
  sys.stdout.write(f"Player {id} enters the game.\n")
  points = 0
  for i in range(THROWS):
    dice = random.randint(DICE_MIN, DICE_MAX)
    points += dice
    sys.stdout.write(f"Player {id} - Throw {i + THROW_INDEX_OFFSET}: {dice}\n")
    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))
  sys.stdout.write(f"Player {id} finished with {points} points.\n")

def main():
  processes = []

  for player_id in range(PLAYER):
    try:
      pid = os.fork()
    except OSError as e:
      sys.exit(f"Error while creating process n° {player_id + PLAYER_ID_OFFSET}: {e}")

    if pid:
      processes.append(pid)
    else:
      player(player_id + PLAYER_ID_OFFSET)
      os._exit(os.EX_OK)

  for pid in processes:
    os.waitpid(pid, WAIT_BLOCKING)

  print("All players have finished")

if __name__ == "__main__":
  main()

Overwriting fork-processes.py




---

**Ejecución**

Se ejecuta el programa

In [ ]:
!python3 fork-processes.py

Player 2 enters the game.
Player 4 enters the game.
Player 4 - Throw 1: 2
Player 2 - Throw 1: 2
Player 3 enters the game.
Player 3 - Throw 1: 2
Player 1 enters the game.
Player 1 - Throw 1: 1
Player 5 enters the game.
Player 5 - Throw 1: 4
Player 5 - Throw 2: 3
Player 4 - Throw 2: 1
Player 2 - Throw 2: 6
Player 3 - Throw 2: 1
Player 1 - Throw 2: 3
Player 5 - Throw 3: 5
Player 2 - Throw 3: 1
Player 3 - Throw 3: 5
Player 4 - Throw 3: 1
Player 1 - Throw 3: 6
Player 1 - Throw 4: 4
Player 5 - Throw 4: 3
Player 2 - Throw 4: 4
Player 3 - Throw 4: 3
Player 4 - Throw 4: 6
Player 3 - Throw 5: 6
Player 1 - Throw 5: 5
Player 5 - Throw 5: 1
Player 2 - Throw 5: 6
Player 4 - Throw 5: 3
Player 5 - Throw 6: 2
Player 3 - Throw 6: 1
Player 1 - Throw 6: 1
Player 2 - Throw 6: 1
Player 4 - Throw 6: 4
Player 3 - Throw 7: 6
Player 2 - Throw 7: 6
Player 5 - Throw 7: 6
Player 1 - Throw 7: 2
Player 2 - Throw 8: 6
Player 4 - Throw 7: 6
Player 1 - Throw 8: 1
Player 3 - Throw 8: 1
Player 2 - Throw 9: 4
Player 5 - T

# Ejercicio 3. Java

**Código**

A continuación se presentan las clases que componen el programa.

Clase principal que lanza los procesos:

In [ ]:
%%writefile MainMonitor.java

import java.io.*;
import java.util.*;

public class MainMonitor
{

  private static final String[] ZONES =
  {
    "Basement", "Attic", "Kitchen",
    "Bedroom", "Garden", "Mausoleum"
  };

  public static void main(String[] args) throws IOException, InterruptedException
  {
    if (args.length != 2)
    {
      System.out.println("Usage: java MainMonitor <duration> <interval>");
      return;
    }

    int duration = Integer.parseInt(args[0]);
    int interval = Integer.parseInt(args[1]);

    List<Process> processes = new ArrayList<>();
    List<BufferedReader> readers = new ArrayList<>();

    for (String zone : ZONES)
    {
      ProcessBuilder builder = new ProcessBuilder
      (
        "java", "CameraProcess", zone,
        String.valueOf(duration),
        String.valueOf(interval)
      );
      builder.redirectErrorStream(true);
      Process process = builder.start();
      processes.add(process);
      readers.add(new BufferedReader(new InputStreamReader(process.getInputStream())));
    }

    boolean finishedCameras = false;
    while (!finishedCameras)
    {
      finishedCameras = true;
      for (int i = 0; i < readers.size(); i++)
      {
        BufferedReader reader = readers.get(i);
        if (reader.ready())
        {
          String line = reader.readLine();
          if (line != null)
          {
            System.out.println(line);
            finishedCameras = false;
          }
        } else
        {
          if (processes.get(i).isAlive())
          {
            finishedCameras = false;
          }
        }
      }
    }

    for (Process process : processes)
    {
      process.waitFor();
    }

    System.out.println("All camera processes have finished.");
  }
}

Overwriting MainMonitor.java


Clase que representa cada cámara individual:

In [ ]:
%%writefile CameraProcess.java

import java.util.Random;

public class CameraProcess
{
  public static final int NO_ACTIVITY = 0;
  public static final int MOVEMENT_DETECTED = 1;
  public static final int THERMAL_ANOMALY = 2;
  public static final int STRANGE_SHADOW = 3;
  public static final int NOISE_DETECTED = 4;

  public static final String[] EVENT_NAMES =
  {
    "No Activity",
    "Movement Detected",
    "Thermal Anomaly",
    "Strange Shadow",
    "Noise Detected"
  };

  public static void main(String[] args)
  {
    if (args.length != 3)
    {
      System.out.println("Usage: java CameraProcess <zone> <duration> <interval>");
      return;
    }

    String zone = args[0];
    int durationSeconds = Integer.parseInt(args[1]);
    int intervalSeconds = Integer.parseInt(args[2]);

    int eventCount = 0;
    Random randomEvent = new Random();

    long endTime = System.currentTimeMillis() + (durationSeconds * 1000);

    while (System.currentTimeMillis() < endTime)
    {
      int eventId = randomEvent.nextInt(5);
      System.out.printf("[CAMERA-%s] Zone: %s | Event: %s%n",
        ProcessHandle.current().pid(),
        zone,
        EVENT_NAMES[eventId]);
      if (eventId != NO_ACTIVITY)
      {
        eventCount++;
      }
      try {
        Thread.sleep(intervalSeconds * 1000);
      } catch (InterruptedException e)
      {
        e.printStackTrace();
      }
    }

    System.out.printf("[CAMERA-%s] Zone: %s | Paranormal events: %d%n",
      ProcessHandle.current().pid(),
      zone,
      eventCount);
  }
}

Overwriting CameraProcess.java




---

**Compilación**

Se compilan ambas clases:

In [ ]:
!javac MainMonitor.java CameraProcess.java



---

**Ejecución**

Se ejecuta el monitor con duración e intervalo respectivamente definidos:

In [ ]:
!java MainMonitor 10 2

[CAMERA-6745] Zone: Basement | Event: Thermal Anomaly
[CAMERA-6757] Zone: Kitchen | Event: Thermal Anomaly
[CAMERA-6764] Zone: Bedroom | Event: Movement Detected
[CAMERA-6753] Zone: Attic | Event: Strange Shadow
[CAMERA-6774] Zone: Garden | Event: No Activity
[CAMERA-6780] Zone: Mausoleum | Event: No Activity
[CAMERA-6745] Zone: Basement | Event: Strange Shadow
[CAMERA-6757] Zone: Kitchen | Event: No Activity
[CAMERA-6764] Zone: Bedroom | Event: Movement Detected
[CAMERA-6753] Zone: Attic | Event: Thermal Anomaly
[CAMERA-6774] Zone: Garden | Event: Noise Detected
[CAMERA-6780] Zone: Mausoleum | Event: Noise Detected
[CAMERA-6745] Zone: Basement | Event: No Activity
[CAMERA-6757] Zone: Kitchen | Event: No Activity
[CAMERA-6764] Zone: Bedroom | Event: No Activity
[CAMERA-6753] Zone: Attic | Event: Strange Shadow
[CAMERA-6774] Zone: Garden | Event: No Activity
[CAMERA-6780] Zone: Mausoleum | Event: Noise Detected
[CAMERA-6745] Zone: Basement | Event: Noise Detected
[CAMERA-6757] Zone: Kit